# Model experiments with MLflow

This notebook runs every recommender model with every data processor, logs the experiments to MLflow, and saves the processed datasets and model artifacts.

It is designed for repeatable experiment tracking and DVC-friendly dataset versioning.

In [1]:
from __future__ import annotations

import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import gc
from torch.utils.data import DataLoader, random_split
import yaml

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from recommender.data import DataProcessorContext, RecommenderDataset, load_events
from recommender.models import ModelFactory
from recommender.mlflow_toolkit import MLflowToolkit
from recommender.training import Trainer, hit_rate_at_k, ndcg_at_k
from recommender.training.early_stopping import EarlyStopping


In [2]:
CONFIG_PATH = ROOT / "configs" / "model.yaml"
MLFLOW_CONFIG_PATH = ROOT / "configs" / "mlflow.yaml"
PROCESSED_DIR = ROOT / "data" / "processed" / "mlflow_experiments"
ARTIFACT_DIR = ROOT / "models" / "mlflow_experiments"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH) as f:
    base_cfg = yaml.safe_load(f)["model"]

with open(MLFLOW_CONFIG_PATH) as f:
    mlflow_cfg = yaml.safe_load(f)["mlflow"]

processor_names = ["weighted", "binary", "implicit"]
model_types = ["ncf", "gmf", "matrix_factorization"]
total_runs = len(processor_names) * len(model_types)

processor_kwargs_map = {
    "weighted": {},
    "binary": {},
    "implicit": {},
}

mlflow_toolkit = MLflowToolkit(
    tracking_uri=mlflow_cfg.get("tracking_uri"),
    experiment_name=mlflow_cfg.get("experiment_name"),
)
mlflow_toolkit.setup()
print(f"MLflow tracking URI: {mlflow_cfg.get('tracking_uri')}")
print(f"Experiment: {mlflow_cfg.get('experiment_name')}")
print(f"Planned runs: {total_runs} ({len(model_types)} models x {len(processor_names)} processors)")

base_cfg


MLflow tracking URI: sqlite:///mlflow.db
Experiment: ecommerce_recommender-dev
Planned runs: 9 (3 models x 3 processors)


{'type': 'ncf',
 'seed': 42,
 'batch_size': 256,
 'learning_rate': 0.001,
 'epochs': 10,
 'show_progress': True,
 'num_negatives': 4,
 'min_interactions': 5,
 'raw_events_path': 'data/raw/events.csv',
 'artifact_dir': 'models',
 'processor': 'weighted',
 'processor_kwargs': {},
 'streaming': False,
 'early_stopping': {'enabled': True,
  'monitor': 'val_loss',
  'mode': 'min',
  'patience': 3,
  'min_delta': 0.0},
 'hyperparams': {'embedding_dim': 64,
  'hidden_layers': [128, 64, 32],
  'dropout': 0.2}}

In [3]:
def build_model_hyperparams(cfg: dict) -> dict:
    hyper = cfg.get("hyperparams", {}) or {}
    keys = ("embedding_dim", "hidden_layers", "dropout", "projection_dim", "global_bias")
    return {k: v for k, v in hyper.items() if k in keys}


def resolve_device() -> str:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        cuda_available = torch.cuda.is_available()
    return "cuda" if cuda_available else "cpu"


def dvc_add(path: Path) -> None:
    try:
        relative_path = path.relative_to(ROOT)
    except ValueError:
        relative_path = path

    try:
        result = subprocess.run(
            ["dvc", "add", str(relative_path)],
            cwd=ROOT,
            check=True,
            capture_output=True,
            text=True,
        )
        print(result.stdout.strip() or f"DVC tracked: {relative_path}")
    except FileNotFoundError:
        print(f"DVC not available, skipped: dvc add {relative_path}")
    except subprocess.CalledProcessError as exc:
        print(f"DVC add failed for {relative_path}: {exc.stderr or exc.stdout}")


def prepare_processor_dataset(events: pd.DataFrame, processor_name: str) -> dict:
    print(f"Preparing processor dataset: {processor_name}")
    processor = DataProcessorContext(processor_name, **processor_kwargs_map[processor_name])
    interactions, user2idx, item2idx = processor.process(
        events,
        min_interactions=base_cfg.get("min_interactions", 1),
    )

    out_path = PROCESSED_DIR / f"{processor_name}_interactions.csv"
    interactions.to_csv(out_path, index=False)
    dvc_add(out_path)
    print(
        f"Saved {processor_name}: rows={len(interactions):,}, "
        f"users={len(user2idx):,}, items={len(item2idx):,}, path={out_path}"
    )

    return {
        "interactions": interactions,
        "user2idx": user2idx,
        "item2idx": item2idx,
        "path": out_path,
    }


def print_training_context(
    run_number: int,
    total_runs: int,
    model_type: str,
    processor_name: str,
    processor_data: dict,
) -> None:
    interactions = processor_data["interactions"]
    print("\n" + "=" * 72)
    print(f"Run {run_number}/{total_runs}")
    print(f"Model:     {model_type}")
    print(f"Dataset:   {processor_name} processor")
    print(f"Data file: {processor_data['path']}")
    print(
        f"Rows:      {len(interactions):,} | "
        f"Users: {len(processor_data['user2idx']):,} | "
        f"Items: {len(processor_data['item2idx']):,}"
    )
    print("=" * 72)


In [4]:
def train_one_experiment(
    processor_data: dict,
    model_type: str,
    processor_name: str,
    seed: int,
) -> dict:

    interactions = processor_data["interactions"]
    user2idx = processor_data["user2idx"]
    item2idx = processor_data["item2idx"]
    processed_path = processor_data["path"]

    print(f"Training model={model_type}, processor={processor_name}")

    mlflow_toolkit.log_dataset(
        interactions,
        name=f"{processor_name}_interactions",
        source=str(processed_path),
        context="training",
    )

    num_users = len(user2idx)
    num_items = len(item2idx)

    dataset = RecommenderDataset(
        interactions,
        num_items,
        num_negatives=base_cfg["num_negatives"],
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(seed),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=base_cfg["batch_size"],
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=base_cfg["batch_size"],
    )

    device = resolve_device()

    print(
        f"Device: {device} | "
        f"samples={len(dataset):,} | "
        f"train={train_size:,} | "
        f"val={val_size:,}"
    )

    model = ModelFactory.create(
        model_type,
        num_users=num_users,
        num_items=num_items,
        **build_model_hyperparams(base_cfg),
    )

    trainer = Trainer(model, base_cfg, device=device)

    early_stopping = EarlyStopping(
        patience=5,
        mode="max",
        min_delta=1e-3,
    )

    # ------------------------------------------------------------
    # MLflow callback (called immediately after every epoch)
    # ------------------------------------------------------------
    def mlflow_logger(epoch_result):
        mlflow_toolkit.log_metrics(
            {
                "train_loss": float(epoch_result.train_loss),
                "learning_rate": float(epoch_result.learning_rate),
                **{
                    k: float(v)
                    for k, v in epoch_result.eval_metrics.items()
                },
            },
            step=epoch_result.epoch,
        )

    history, best = trainer.fit_with_early_stopping(
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=base_cfg["epochs"],
        early_stopping=early_stopping,
        monitor="auc_roc",
        show_progress=base_cfg.get("show_progress", True),
        log_callback=mlflow_logger,
    )

    # ------------------------------------------------------------
    # Metrics from BEST epoch
    # ------------------------------------------------------------
    best_result = next(
        r for r in history if r.epoch == best["epoch"]
    )

    best_loss = float(best_result.train_loss)
    best_metrics = best_result.eval_metrics

    mlflow_toolkit.log_metrics(
        {
            "best_epoch": int(best["epoch"]),
            "best_auc_roc": float(best["value"]),
            "epochs_run": len(history),
        }
    )

    # ------------------------------------------------------------
    # Ranking metrics (model already restored to best weights)
    # ------------------------------------------------------------
    val_indices = val_dataset.indices

    val_samples = np.array(
        [dataset.samples[i] for i in val_indices[: min(10000, len(val_indices))]]
    )

    positive_only = (
        val_samples[val_samples[:, 2] == 1.0][:, :2]
        .astype(np.int64)
    )

    hr = hit_rate_at_k(
        model,
        positive_only[:1000],
        num_items,
        k=10,
        device=device,
    )

    ndcg = ndcg_at_k(
        model,
        positive_only[:1000],
        num_items,
        k=10,
        device=device,
    )

    checkpoint_path = ARTIFACT_DIR / f"{model_type}_{processor_name}.pt"

    torch.save(
        {
            "model_type": model_type,
            "processor": processor_name,
            "model_state_dict": model.state_dict(),
            "user2idx": user2idx,
            "item2idx": item2idx,
            "config": base_cfg,
            "metrics": {
                "loss": best_loss,
                "auc_roc": float(best_metrics["auc_roc"]),
                "avg_precision": float(best_metrics["avg_precision"]),
                "hit_rate_at_10": float(hr),
                "ndcg_at_10": float(ndcg),
            },
            "early_stopping": {
                "best_epoch": best["epoch"],
                "best_auc_roc": best["value"],
                "epochs_run": len(history),
            },
        },
        checkpoint_path,
    )

    mlflow_toolkit.log_artifact(checkpoint_path)

    mlflow_toolkit.log_metrics(
        {
            "final_train_loss": best_loss,
            "final_auc_roc": float(best_metrics["auc_roc"]),
            "final_avg_precision": float(best_metrics["avg_precision"]),
            "hit_rate_at_10": float(hr),
            "ndcg_at_10": float(ndcg),
        }
    )

    print(f"Saved model artifact: {checkpoint_path}")

    result = {
        "model_type": model_type,
        "processor": processor_name,
        "artifact": str(checkpoint_path),
        "processed_data": str(processed_path),
        "train_loss": best_loss,
        "auc_roc": float(best_metrics["auc_roc"]),
        "avg_precision": float(best_metrics["avg_precision"]),
        "hit_rate_at_10": float(hr),
        "ndcg_at_10": float(ndcg),
        "best_epoch": best["epoch"],
        "epochs_run": len(history),
    }

    del trainer
    del model
    del train_loader
    del val_loader
    del train_dataset
    del val_dataset
    del dataset

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    return result

In [5]:
raw_events_path = ROOT / base_cfg.get("raw_events_path", "data/raw/events.csv")
print(f"Loading events from: {raw_events_path}")
events = load_events(str(raw_events_path))
print(f"Loaded events: {len(events):,}")

processor_cache = {
    processor_name: prepare_processor_dataset(events, processor_name)
    for processor_name in processor_names
}

results = []
run_number = 0
for model_type in model_types:
    for processor_name in processor_names:
        run_number += 1
        run_name = f"{model_type}-{processor_name}"
        print(f"=== Run {run_number}/{total_runs}: {run_name} ===")
        with mlflow_toolkit.start_run(
            run_name=run_name,
            tags={"model_type": model_type, "processor": processor_name},
        ):
            mlflow_toolkit.log_params({
                "model_type": model_type,
                "processor": processor_name,
                "seed": base_cfg["seed"],
                "batch_size": base_cfg["batch_size"],
                "learning_rate": base_cfg["learning_rate"],
                "epochs": base_cfg["epochs"],
                "num_negatives": base_cfg["num_negatives"],
                "min_interactions": base_cfg.get("min_interactions", 1),
            })
            result = train_one_experiment(
                processor_cache[processor_name],
                model_type,
                processor_name,
                seed=base_cfg["seed"],
            )
            results.append(result)

results_df = pd.DataFrame(results)
results_df.sort_values(["model_type", "processor"]).reset_index(drop=True)


Loading events from: /home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/events.csv
Loaded events: 2,756,101
Preparing processor dataset: weighted
DVC tracked: data/processed/mlflow_experiments/weighted_interactions.csv
Saved weighted: rows=897,028, users=81,318, items=67,625, path=/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/processed/mlflow_experiments/weighted_interactions.csv
Preparing processor dataset: binary
DVC tracked: data/processed/mlflow_experiments/binary_interactions.csv
Saved binary: rows=21,796, users=2,243, items=4,546, path=/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/processed/mlflow_experiments/binary_interactions.csv
Preparing processor dataset: implicit
DVC tracked: data/processed/mlflow_experiments/implicit_interactions.csv
Saved implicit: rows=897,028, users=81,318, items=67,625, path=/home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/data/processed/mlflow_experiments/implicit_

/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Device: cuda | samples=4,485,140 | train=3,588,112 | val=897,028


Epoch 01/10 | loss=0.3912 | auc=0.8473 | ap=0.6720


Epoch 02/10 | loss=0.2899 | auc=0.8779 | ap=0.7448


...

Epoch 09/10 | loss=0.0692 | auc=0.9075 | ap=0.8182


Epoch 10/10 | loss=0.0601 | auc=0.9081 | ap=0.8209
Saved model artifact: /home/gusdpaula/code-dev/MLENG_FIAP/fase_2/ecommerce_recommender/models/mlflow_experiments/ncf_implicit.pt
=== Run 4/9: gmf-weighted ===
Training model=gmf, processor=weighted


/home/gusdpaula/.cache/pypoetry/virtualenvs/ecommerce-recommender--azJ085T-py3.12/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Device: cuda | samples=4,485,140 | train=3,588,112 | val=897,028


TypeError: GMFModel.__init__() got an unexpected keyword argument 'hidden_layers'

## What gets saved

For each processor this notebook produces one dataframe saved under `data/processed/mlflow_experiments/` and tracked with `dvc add` when DVC is available.

For each `(model, processor)` run it logs MLflow params, per-epoch metrics, final ranking metrics, the input dataset reference, and the PyTorch checkpoint under `models/mlflow_experiments/`.

The progress output shows dataset preparation, run number, device, sample counts, per-epoch metrics, and final artifact paths so long-running experiments do not feel like a black box.
